
# Ollama

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="deepseek-r1:7b",
    temperature=0,
)

In [2]:
from langchain_core.messages import AIMessage

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='<think>\nOkay, so I need to translate "I love programming." into French. Let me think about how to approach this.\n\nFirst, "I love" is straightforward in French. The word for "love" is "amour," and since it\'s a present tense verb, we use the third person singular form: "J\' adore." Wait, no, actually, "I love" would be "J\'aime" because "adore" is more of a passionate love. Hmm, but sometimes people just say "J\'adore" even if it\'s not as strong. I think in this context, since it\'s just expressing love without specifying the intensity, maybe "J\'aime" is better.\n\nNext, "programming." The word for programming in French is "programmation." So putting it all together, it would be "J\'aime le programming." But wait, does that sound natural? Let me check. Maybe "programmer" is more commonly used than "programmation"? No, actually, "programmation" refers to the act of programming itself, while "programmer" means to program. So in this context, since we\'re talking ab

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

AIMessage(content='<think>\nOkay, so I need to translate the sentence "I love programming." into German. Let me think about how to approach this.\n\nFirst, "I love" is an expression of emotion in English. In German, there\'s no direct equivalent, but we can use "Ich liebe" which means "I love." So that part should be straightforward.\n\nNext, the object of the verb is "programming." Now, I need to figure out how to translate "programming" into German. The word "programmieren" comes to mind, which means "to program." But since it\'s an object here, not a verb, maybe we just use the noun form? So perhaps "Programmieren" with a preposition.\n\nWait, actually, in this context, when you say "I love programming," it\'s more like "Ich love to program." In German, that would be "Ich liebe zu programmieren." But I\'m not sure if "zu" is necessary here. Sometimes people just say "Ich love programming" as "Ich liebe Programming."\n\nBut wait, in German, when you use the verb "liebe," which means 

In [ ]:
!ollama pull llama3.1

In [9]:
from typing import List

from langchain_core.tools import tool

@tool
def validate_user(user_id: int, addresses: List[str]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id (int): the user ID.
        addresses (List[str]): Previous addresses as a list of strings.
    """
    return True


llm = ChatOllama(
    model="llama3.1",
    temperature=0,
).bind_tools([validate_user])

result = llm.invoke(
    "Could you validate user 123? They previously lived at "
    "123 Fake St in Boston MA and 234 Pretend Boulevard in "
    "Houston TX."
)
result.tool_calls

[{'name': 'validate_user',
  'args': {'addresses': ['123 Fake St, Boston, MA',
    '234 Pretend Boulevard, Houston, TX'],
   'user_id': 123},
  'id': '9c0734cf-dd64-4d0b-bc7d-e50f0ad9c392',
  'type': 'tool_call'}]

In [12]:
@tool
async def amultiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

for att in dir(amultiply):
    if not att.startswith("_"):
        print(att)

InputType
OutputType
abatch
abatch_as_completed
ainvoke
args
args_schema
arun
as_tool
assign
astream
astream_events
astream_log
atransform
batch
batch_as_completed
bind
callback_manager
callbacks
config_schema
config_specs
configurable_alternatives
configurable_fields
construct
copy
coroutine
description
dict
from_function
from_orm
func
get_config_jsonschema
get_graph
get_input_jsonschema
get_input_schema
get_lc_namespace
get_name
get_output_jsonschema
get_output_schema
get_prompts
handle_tool_error
handle_validation_error
input_schema
invoke
is_lc_serializable
is_single_input
json
lc_attributes
lc_id
lc_secrets
map
metadata
model_computed_fields
model_config
model_construct
model_copy
model_dump
model_dump_json
model_extra
model_fields
model_fields_set
model_json_schema
model_parametrized_name
model_post_init
model_rebuild
model_validate
model_validate_json
model_validate_strings
name
output_schema
parse_file
parse_obj
parse_raw
pick
pipe
raise_deprecation
response_format
return_direc

In [16]:
from typing import Annotated, List
import pprint

@tool
def multiply_by_max(
    a: Annotated[int, "scale factor"],
    b: Annotated[List[int], "list of ints over which to take maximum"],
) -> int:
    """Multiply a by the maximum of b."""
    return a * max(b)


pprint.pprint(multiply_by_max.args_schema.model_json_schema())

{'description': 'Multiply a by the maximum of b.',
 'properties': {'a': {'description': 'scale factor',
                      'title': 'A',
                      'type': 'integer'},
                'b': {'description': 'list of ints over which to take maximum',
                      'items': {'type': 'integer'},
                      'title': 'B',
                      'type': 'array'}},
 'required': ['a', 'b'],
 'title': 'multiply_by_max',
 'type': 'object'}


In [17]:
from pprint import pp
from pydantic import BaseModel, Field


class CalculatorInput(BaseModel):
    a: int = Field(description="first number")
    b: int = Field(description="second number")


@tool("multiplication-tool", args_schema=CalculatorInput, return_direct=True)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

pprint.pprint(multiply.args_schema.model_json_schema())

{'properties': {'a': {'description': 'first number',
                      'title': 'A',
                      'type': 'integer'},
                'b': {'description': 'second number',
                      'title': 'B',
                      'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'CalculatorInput',
 'type': 'object'}


In [18]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [("human", "Hello. Please respond in the style of {answer_style}.")]
)

# Placeholder LLM
llm = GenericFakeChatModel(messages=iter(["hello matey"]))

chain = prompt | llm | StrOutputParser()

as_tool = chain.as_tool(
    name="Style responder", description="Description of when to use tool."
)
as_tool.args

/var/folders/54/_pbcgv8n7f1flbc1s4yw819m0000gn/T/ipykernel_46481/2548361071.py:14: LangChainBetaWarning: This API is in beta and may change in the future.
  as_tool = chain.as_tool(


{'answer_style': {'title': 'Answer Style', 'type': 'string'}}

# HuggingFace

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import LLMChain

repo_id = "mistralai/Mistral-7B-Instruct-v0.3"
llm = HuggingFaceEndpoint(
    repo_id=repo_id, 
    max_length=128, 
    temperature=0.5,
    huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_TOKEN")
)

template = """Question: {question} Answer: Let's think step by step."""
prompt = PromptTemplate.from_template(template)

llm_chain = prompt | llm | StrOutputParser()

WARNING! max_length is not default parameter.
                    max_length was transferred to model_kwargs.
                    Please make sure that max_length is what you intended.


In [12]:
llm.invoke("What is the capital of France?")

/Users/huangtang/CodeRepo/generative_ai_with_langchain/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


ValueError: Task unknown has no recommended model. Please specify a model explicitly. Visit https://huggingface.co/tasks for more info.

In [9]:
llm_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template="Question: {question} Answer: Let's think step by step.")
| HuggingFaceEndpoint(repo_id='mistralai/Mistral-7B-Instruct-v0.2', temperature=0.5, stop_sequences=[], server_kwargs={}, model_kwargs={'max_length': 128}, model='mistralai/Mistral-7B-Instruct-v0.2', client=<InferenceClient(model='mistralai/Mistral-7B-Instruct-v0.2', timeout=120)>, async_client=<InferenceClient(model='mistralai/Mistral-7B-Instruct-v0.2', timeout=120)>)
| StrOutputParser()

In [ ]:
question = "Who won the FIFA World Cup in 1994?"
print(llm_chain.invoke({'question': question}))

/Users/huangtang/CodeRepo/generative_ai_with_langchain/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


ValueError: Task unknown has no recommended model. Please specify a model explicitly. Visit https://huggingface.co/tasks for more info.